In [1]:
import sys,os
sys.path.append(r'C:/Users/andrej/Projects/EnergyTrading/Python/Strategies/LeadLagXGB/')
sys.path.append(r'Z:/EnergyTrading/Python/')
sys.path.append(r'Z:/EnergyTrading/Python/Strategies/LeadLagXGB/')
from support_functions import calculate_MACD, calculate_lead_lag_triggers, calculate_regression_model_price, calc_vol_intensity_index

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

In [3]:
def get_trades_for_contract(contract, start_date, end_date):
    params_dict={}
    params_dict['tenor_list'] = ['dec'] if contract=='euadec1' else [contract[-2]]
    params_dict['tn1_list'] = [int(contract[-1])]
    params_dict['mkt_list'] = ['eua'] * len(params_dict['tenor_list']) if contract=='euadec1' else [contract[0:-2]] * len(params_dict['tenor_list'])
    params_dict['tn2_list'] = []
    params_dict['prod'] = 'base'
    params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
    params_dict['start_date'] = start_date
    params_dict['end_date'] = end_date
    params_dict['ns'] = 2

    # Fetch trades and best orders for the curve
    assembler = TPDataAssembly(source='database', user='matej')
    # assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
    trades_dict = assembler.get_data(params_dict, target_data='trades')
    #assembler.set_data_source('database')
    #ba_dict = assembler.get_data(params_dict, target_data='best_orders')


    trades = pd.DataFrame()
    products = []
    for key in trades_dict.keys():
        trade_aux = trades_dict[key].copy()
        trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
        if trades.empty:
            trades = trade_aux.copy()
        else:
            trades = pd.concat([trades, trade_aux])
        products.append(key)
    trades.sort_index(inplace=True)




    data_raw = trades
    print(data_raw.columns)
    data_raw['tradeid_'+contract]=data_raw['tradeid_'+contract].apply(lambda x: str(x)[:-7] if str(x)[-7:]==' Public' else str(x))
    df_lead = data_raw[['tradeid_'+contract,'price_'+contract, 'volume_'+contract]].copy()
    
    df_lead['contract']=contract

    # data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
    #                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

    df_lead.columns = [a.split('_')[0] for a in df_lead.columns]
    print(df_lead.columns)
    df_lead.columns = ['tradeid', 'trd_price', 'volume', 'contract']
    
    return df_lead

In [4]:
start_date='2025-01-01'
end_date='2025-05-31'

In [5]:
'esw1', 'esw2', 'esm1', 'esm2', 'esm3', 'esq1', 'esq2', 'esq3', 'esq4', 'esy1', 'esy2'

('esw1',
 'esw2',
 'esm1',
 'esm2',
 'esm3',
 'esq1',
 'esq2',
 'esq3',
 'esq4',
 'esy1',
 'esy2')

# Selecting all lead and lag EEX trades since 2024 -> this is the base for predictor calculations

In [6]:
contracts=['dew1', 'dew2', 'dem1', 'dem2', 'dem3', 'deq1', 'deq2', 'deq3', 'deq4', 'dey1', 'dey2',
          'frw1', 'frw2', 'frm1', 'frm2', 'frm3', 'frq1', 'frq2', 'frq3', 'frq4', 'fry1', 'fry2',
          'itw1', 'itw2', 'itm1', 'itm2', 'itm3', 'itq1', 'itq2', 'itq3', 'itq4', 'ity1', 'ity2',
          'esw1', 'esw2', 'esm1', 'esm2', 'esm3', 'esq1', 'esq2', 'esq3', 'esq4', 'esy1', 'esy2',
          'ttfm1', 'ttfq1']
contracts

['dew1',
 'dew2',
 'dem1',
 'dem2',
 'dem3',
 'deq1',
 'deq2',
 'deq3',
 'deq4',
 'dey1',
 'dey2',
 'frw1',
 'frw2',
 'frm1',
 'frm2',
 'frm3',
 'frq1',
 'frq2',
 'frq3',
 'frq4',
 'fry1',
 'fry2',
 'itw1',
 'itw2',
 'itm1',
 'itm2',
 'itm3',
 'itq1',
 'itq2',
 'itq3',
 'itq4',
 'ity1',
 'ity2',
 'esw1',
 'esw2',
 'esm1',
 'esm2',
 'esm3',
 'esq1',
 'esq2',
 'esq3',
 'esq4',
 'esy1',
 'esy2',
 'ttfm1',
 'ttfq1']

In [7]:
df_all=pd.concat([get_trades_for_contract(contract, start_date, end_date) for contract in contracts])

Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connec

Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Index(['tradeid_frw1', 'price_frw1', 'volume_frw1', 'action_frw1',
       'broker_id_frw1', 'own_trades_frw1'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from t

Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Problem in code:  'Index' object has no attribute 'microsecond'
Connected to the database oracle


C:\Users\andrej\Projects\EnergyTrading\Python\Database\TPData.py:571: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_out.index = df_out.index + pd.to_timedelta(df_data.loc[~idx, 'nanotime'].astype(float), unit='ns')


Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Problem in code:  'Index' object has no attribute 'microsecond'
Connected to the database oracle


...

Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Problem in code:  'Index' object has no attribute 'microsecond'
Connected to the database oracle


C:\Users\andrej\Projects\EnergyTrading\Python\Database\TPData.py:571: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_out.index = df_out.index + pd.to_timedelta(df_data.loc[~idx, 'nanotime'].astype(float), unit='ns')


Disconnected from the database oracle
Problem in code:  'Index' object has no attribute 'microsecond'
Index(['tradeid_esw2', 'price_esw2', 'volume_esw2', 'action_esw2',
       'broker_id_esw2', 'own_trades_esw2'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')
Connected to the database oracle


C:\Users\andrej\Projects\EnergyTrading\Python\Database\TPData.py:571: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_out.index = df_out.index + pd.to_timedelta(df_data.loc[~idx, 'nanotime'].astype(float), unit='ns')


Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Index(['tradeid_esm1', 'price_esm1', 'volume_esm1', 'action_esm1',
       'broker_id_esm1', 'own_trades_esm1'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from t

In [8]:
df_all['contract'].value_counts()

contract
dem1     354917
ttfm1    138177
dem2     125544
deq1     111020
dew1      82196
dey1      71418
frm1      63456
deq2      58643
dew2      43841
frm2      28000
deq3      26533
dem3      21990
frq1      20551
itm1      18778
frw1      17150
ttfq1     15156
dey2      11516
itq1      10329
frq2      10176
deq4       9736
fry1       8454
esm1       7011
itq2       6460
itm2       5647
frw2       5386
frq3       4276
fry2       4093
ity1       4000
frm3       3909
esq1       3305
frq4       2840
esm2       2643
esq2       2461
itq3       2331
esq3       1257
esy1       1107
esw1       1104
itw1        889
esm3        872
itm3        534
itq4        409
ity2        386
esy2        365
esw2        220
esq4        218
itw2        213
Name: count, dtype: int64

In [9]:
df_all['datetime'] = pd.to_datetime(df_all.index, errors='coerce')
df_all['timestamp'] = df_all['datetime']

In [10]:
df_all=df_all[df_all['datetime'].apply(lambda x: x.hour>=8 and  x.hour<18)]

In [11]:
df_all.head()

,tradeid,trd_price,volume,contract,datetime,timestamp
2025-01-02 08:21:52.335000000,7285750,120.5,5,dew1,2025-01-02 08:21:52.335000000,2025-01-02 08:21:52.335000000
2025-01-02 08:21:52.353397962,Eurex T7/DEB2012025-20250102/149/35,120.5,35,dew1,2025-01-02 08:21:52.353397962,2025-01-02 08:21:52.353397962
2025-01-02 08:21:52.451683988,Eurex T7/DEB2012025-20250102/150/25,120.5,25,dew1,2025-01-02 08:21:52.451683988,2025-01-02 08:21:52.451683988
2025-01-02 08:21:52.464362395,Eurex T7/DEB2012025-20250102/151/2,120.5,2,dew1,2025-01-02 08:21:52.464362395,2025-01-02 08:21:52.464362395
2025-01-02 08:21:52.477104359,Eurex T7/DEB2012025-20250102/152/25,120.5,25,dew1,2025-01-02 08:21:52.477104359,2025-01-02 08:21:52.477104359


In [12]:
df_all['datetime'] = df_all['datetime'].astype('datetime64[us]')


In [13]:
# Insert into targets_dataset_entries
conn = Database('timescaledb')
conn._connect()
df_all[['datetime','tradeid','contract']].to_sql('tagged_trades_with_contract', conn.engine, schema='public', index=False, if_exists='append', method='multi')

Connected to the database timescaledb


1309494